<div style='background-color:#002147; padding:20px; border-radius:8px'>
<h1 style='color:white;'>Cálculo Numérico</h1>
<h3 style='color:white;'>Integração Numérica</h3>
<h4 style='color:white;'>Regras do Trapézio e de Simpson</h4>
</div>

## $ \S 0 $ Preparação do ambiente

Execute a célula abaixo para carregar as bibliotecas utilizadas neste Jupyter Notebook.

Usaremos `numpy` para os cálculos numéricos e `matplotlib` para visualizar a interpretação geométrica da integral como área.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Math, display

<div style="background-color:#fff3b0; padding:12px; border-radius:6px;">

**Atenção:** Em Python, os índices começam em 0.  
Nas fórmulas matemáticas, manteremos a notação usual $x_0,x_1,\ldots,x_n$ para os pontos da partição.

</div>

## $ \S 1 $ Motivação

A integral definida

$$
I=\int_a^b f(x)\,dx
$$

representa, geometricamente, a área orientada entre o gráfico de $f(x)$ e o eixo $x$ no intervalo $[a,b]$.

A integração numérica é usada quando:

- a primitiva de $f(x)$ é difícil de obter;
- a função é conhecida apenas por uma tabela de valores;
- queremos uma aproximação computacional rápida para uma integral definida.

A ideia central dos métodos desta aula é substituir a curva por polinômios simples em pequenos intervalos.

In [ ]:
# Interpretação geométrica da integral como área.
def f_motivacao(x):
    return np.cos(x)

x = np.linspace(0, 0.8, 300)
y = f_motivacao(x)

plt.figure(figsize=(8, 4.5))
plt.plot(x, y, color="#002147", linewidth=2, label=r"$f(x)=\cos(x)$")
plt.fill_between(x, y, color="#7aa6d8", alpha=0.45, label="área aproximada pela integral")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel(r"$x$")
plt.ylabel(r"$f(x)$")
plt.title(r"área associada a $\int_0^{0.8}\cos(x)\,dx$")
plt.grid(True, linestyle=":", alpha=0.45)
plt.legend()
plt.show()

## $ \S 2 $ Regra do Trapézio

No intervalo $[a,b]$, aproximamos a curva por uma reta que liga os pontos $(a,f(a))$ e $(b,f(b))$.

A área aproximada é a área de um trapézio:

$$
\int_a^b f(x)\,dx \approx I_T = \frac{b-a}{2}\,[f(a)+f(b)].
$$

Verifique o código Python a seguir: `trapezio`. 

In [ ]:
def trapezio(f, a, b, detalhar=True):
    """
    Regra do trapézio simples.
    """
    h = b - a
    soma = f(a) + f(b)
    valor = (h / 2) * soma

    if detalhar:
        print("Regra do trapézio simples")
        print(f"a = {a:g}, b = {b:g}, h = b-a = {h:g}")
        print(f"f(a) = {f(a):.10g}, f(b) = {f(b):.10g}")
        print(f"Integral aproximada = {valor:.10g}")

    return valor

In [ ]:
# Exemplo com a função do roteiro: integral de cos(x) de 0 a 0.8.
valor_trapezio = trapezio(np.cos, 0, 0.8)
valor_exato = np.sin(0.8) - np.sin(0)
print(f"Valor exato = {valor_exato:.10g}")
print(f"Erro absoluto = {abs(valor_exato - valor_trapezio):.3e}")

## $ \S 3 $ Regra do Trapézio Repetida

Para melhorar a aproximação, dividimos $[a,b]$ em $n$ subintervalos de largura

$$
h=\frac{b-a}{n}.
$$

Com os pontos

$$
x_i=a+ih,\qquad i=0,1,\ldots,n,
$$

a regra do trapézio repetida é

$$
\int_a^b f(x)\,dx \approx \frac{h}{2}\left[f(x_0)+f(x_n)+2\sum_{i=1}^{n-1}f(x_i)\right]=T_{n}.
$$

Essa é a versão composta implementada no código `trapezio_repetida`.

In [ ]:
def trapezio_repetida(f, a, b, n, detalhar=True):
    """
    Regra do trapézio repetida, adaptada do script trapezio_repetida.sci.
    """
    if n <= 0:
        raise ValueError("n deve ser um inteiro positivo.")

    h = (b - a) / n
    pontos = np.array([a + i*h for i in range(n + 1)], dtype=float)
    valores = f(pontos)
    soma = valores[0] + valores[-1] + 2*np.sum(valores[1:-1])
    integral = (h/2) * soma

    if detalhar:
        print("Regra do trapézio repetida")
        print(f"a = {a:g}, b = {b:g}, n = {n}, h = {h:g}")
        print(f"f(x_0) = {valores[0]:.10g}, f(x_n) = {valores[-1]:.10g}")
        print(f"Soma ponderada = {soma:.10g}")
        print(f"Integral aproximada = {integral:.10g}")

    return integral

✏️[__Exercício 6-03.1__](#exercicio-6-03.1):  Exercício 1 — comparação para diferentes valores de $h$

Compare os resultados do método do trapézio repetido aplicado à integral

$$
\int_0^{0.8}\cos(x)\,dx,
$$

nos casos:

- $h=0.1$;
- $h=0.2$;
- $h=0.4$.

Como $h=(b-a)/n$, usamos

$$
n=\frac{b-a}{h}.
$$

In [ ]:
a, b = 0, 0.8
valor_exato = np.sin(b) - np.sin(a)

print(f"Valor exato: {valor_exato:.10f}\n")
print(" h      n      T_n           erro absoluto")
print("-"*48)

for h in [0.1, 0.2, 0.4]:
    n = int(round((b-a)/h))
    aproximacao = trapezio_repetida(np.cos, a, b, n, detalhar=False)
    erro = abs(valor_exato - aproximacao)
    print(f"{h:<6g} {n:<6d} {aproximacao:<13.10f} {erro:.3e}")

<div style="background-color:#f0f0f0; padding:12px; border-radius:6px;">

**Observação:** quando $h$ diminui, o número de subintervalos aumenta e a aproximação tende a melhorar. Esse comportamento é esperado porque a curva é aproximada por segmentos menores.

</div>

## $ \S 4 $ Regra $1/3$ de Simpson 

Na regra de Simpson simples, aproximamos $f(x)$ por uma parábola que passa por três pontos:

$$
x_0=a,\qquad x_1=\frac{a+b}{2},\qquad x_2=b.
$$

Tomando

$$
h=\frac{b-a}{2},
$$

a aproximação é

$$
S=\frac{h}{3}\left[f(a)+4f\left(\frac{a+b}{2}\right)+f(b)\right].
$$

In [ ]:
def simpson(f, a, b, detalhar=True):
    """
    Regra de Simpson 1/3 simples.
    """
    h = (b - a) / 2
    m = (a + b) / 2
    soma = f(a) + 4*f(m) + f(b)
    integral = (h/3) * soma

    if detalhar:
        print("Regra de Simpson 1/3 simples")
        print(f"a = {a:g}, m = {m:g}, b = {b:g}, h = {h:g}")
        print(f"f(a) = {f(a):.10g}, f(m) = {f(m):.10g}, f(b) = {f(b):.10g}")
        print(f"Soma ponderada = {soma:.10g}")
        print(f"Integral aproximada = {integral:.10g}")

    return integral

In [ ]:
valor_simpson = simpson(np.cos, 0, 0.8)
valor_exato = np.sin(0.8)
print(f"Valor exato = {valor_exato:.10g}")
print(f"Erro absoluto = {abs(valor_exato - valor_simpson):.3e}")

## $ \S 5 $ Regra de Simpson Repetida

Quando o intervalo é dividido em $n$ subintervalos de mesma largura, com $n$ par, a regra de Simpson repetida é

$$
S_n=\frac{h}{3}\left[f(x_0)+4\sum_{i=1,\;i\;\text{ímpar}}^{n-1}f(x_i)+2\sum_{i=2,\;i\;\text{par}}^{n-2}f(x_i)+f(x_n)\right].
$$

<div style="background-color:#fff3b0; padding:12px; border-radius:6px;">

**Condição importante:** a regra de Simpson repetida exige número par de subintervalos, isto é, $n$ deve ser par.

</div>

In [ ]:
def simpson_repetida(f, a, b, N, detalhar=True):
    """
    Retorna uma aproximação para a integral definida de a a b
    de uma função real f de uma variável usando a regra de Simpson
    composta com N subdivisões do intervalo. N deve ser par!
    """

    if not (isinstance(N, int) and N % 2 == 0 and N >= 2):
        raise ValueError("N deve ser um natural par!")

    h = (b - a) / N  # tamanho do passo
    soma_impares = 0.0
    soma_pares = 0.0
    x = a  # valor atual de x (x_i)

    for i in range(2, N, 2):    # para i = 2, 4, 6, ..., N - 2:
        x += h                  # tome x = a + (i - 1) * h
        soma_impares += f(x)
        x += h                  # tome x = a + i * h
        soma_pares += f(x)

    # f(x_{N-1}) não foi computado acima
    soma_impares_val = soma_impares + f(x + h)
    soma_impares_weighted = 4 * soma_impares_val
    soma_pares_weighted = 2 * soma_pares

    soma = f(a) + f(b) + soma_pares_weighted + soma_impares_weighted
    integral = soma * h / 3.0

    if detalhar:
        print("Regra de Simpson 1/3 repetida")
        print(f"a = {a:g}, b = {b:g}, N = {N}, h = {h:g}")
        print(f"Integral aproximada = {integral:.10g}")

    return integral

In [ ]:
a, b = 0, 0.8
valor_exato = np.sin(b) - np.sin(a)

print(f"Valor exato: {valor_exato:.10f}\n")
print(" n      S_n           erro absoluto")
print("-"*40)

for n in [2, 4, 8]:
    aproximacao = simpson_repetida(np.cos, a, b, n)
    erro = abs(valor_exato - aproximacao)
    print(f"{n:<6d} {aproximacao:<13.10f} {erro:.3e}")

✏️[__Exercício 6-04.1__](#exercicio-6-04.1): Paraquedista

Deseja-se obter a distância percorrida por um paraquedista em queda livre. A velocidade do modelo com resistência do ar é dada por

$$
v(t)=\frac{gm}{c}\left(1-e^{-(c/m)t}\right),
$$

logo a distância percorrida no intervalo $[0,10]$ é

$$
d=\int_0^{10} v(t)\,dt.
$$

Use os dados do roteiro (unidades SI):

$$
g=9.81\ \text{m/s}^2,\qquad m=70\ \text{kg},\qquad c=12\ \text{kg/s}.
$$

Aqui, $t$ está em segundos, $v(t)$ em metros por segundo e $d$ em metros.

✏️[__Exercício 6-04.2__](#exercicio-6-04.2):A função
$$
\text{erf(x)} = \frac{2}{\sqrt\pi}\int_0^x e^{-t^2}\,dt
$$
é chamada de *função erro (de Gauss)*. Ela é bastante importante em Estatística,
mas não é uma função elementar, ou seja, não é possível encontrar uma "fórmula
fechada" para $ \text{erf} $.

(a) Mostre que $ \text{erf}(-x) = -\text{erf}(x) $, isto é, $ \text{erf} $ é uma
função ímpar. *Dica:* Mostre mais geralmente que se $ f $ é uma função par,
então $ g(x) = \int_0^x f(t)\,dt $ é ímpar.

(b) O limite de $ \text{erf}(x) $ conforme $ x \to +\infty $ existe. Obtenha uma
estimativa para ele usando a regra de Simpson da seguinte maneira:
* Transforme a integral imprópria de $ 0 $ a $ +\infty $ numa integral de $ 0 $ a $ 1 $ através da substituição $ t = \frac{u}{1 - u} $.
* Usando a regra de Simpson (com ajuda do computador), estime a integral resultante de
$ 0 $ a $ 1 - \varepsilon $ para $ \varepsilon > 0 $ cada vez menor.

_Solução:_

## $ \S 6 $ Integração a partir de dados tabelados

Em algumas situações, não conhecemos a expressão algébrica de $f(x)$. Temos apenas uma tabela:

$$
\begin{array}{c|ccccc}
x & x_0 & x_1 & x_2 & \cdots & x_n\\
\hline
f(x) & f(x_0) & f(x_1) & f(x_2) & \cdots & f(x_n)
\end{array}
$$

Nesse caso, podemos aplicar as regras diretamente aos valores da tabela, desde que os pontos estejam igualmente espaçados.

In [ ]:
def verificar_espacamento_uniforme(x):
    h = np.diff(x)
    if len(h) == 0:
        raise ValueError("Forneça pelo menos dois pontos.")
    if not np.allclose(h, h[0]):
        raise ValueError("Os pontos x devem estar igualmente espaçados.")
    return h[0]


def trapezio_tabela(x, fx, detalhar=True):
    x = np.asarray(x, dtype=float)
    fx = np.asarray(fx, dtype=float)
    if len(x) != len(fx):
        raise ValueError("x e fx devem ter o mesmo tamanho.")
    h = verificar_espacamento_uniforme(x)
    n = len(x) - 1
    soma = fx[0] + fx[-1] + 2*np.sum(fx[1:-1])
    integral = (h/2) * soma

    if detalhar:
        print("Trapézio repetido para dados tabelados")
        print(f"n = {n}, h = {h:g}")
        print(f"Integral aproximada = {integral:.10g}")

    return integral


def simpson_tabela(x, fx, detalhar=True):
    x = np.asarray(x, dtype=float)
    fx = np.asarray(fx, dtype=float)
    if len(x) != len(fx):
        raise ValueError("x e fx devem ter o mesmo tamanho.")
    h = verificar_espacamento_uniforme(x)
    n = len(x) - 1
    if n % 2 != 0:
        raise ValueError("Para Simpson repetida, o número de subintervalos deve ser par.")

    soma = fx[0] + 4*np.sum(fx[1:-1:2]) + 2*np.sum(fx[2:-1:2]) + fx[-1]
    integral = (h/3) * soma

    if detalhar:
        print("Simpson repetida para dados tabelados")
        print(f"n = {n}, h = {h:g}")
        print(f"Integral aproximada = {integral:.10g}")

    return integral

✏️[__Exercício 6-06.1__](#exercicio-6-6.1) Estime $$\int_0^{3.5} g(x)\,dx $$ a partir dos dados abaixo:

| $ x $    | $0$ |$0.5$|$1.0$|$1.5$ |$2.0$ |$2.5$ |$3.0$ |$3.5$ |
| ---      | --- | --- | --- | ---  | ---  | ---  | ---  | ---  |
| $ g(x) $ |$1.5$|$2.0$|$2.0$|$1.63$|$1.25$|$0.95$|$0.79$|$0.68$|

*Dica:* Crie um procedimento que, dada uma lista de valores $ y_i = f(x_i) $ para $ i = 0, \dots, N $ (com $ x_i $ assumidos igualmente espaçados), calcula a expressão na regra do trapézio.

## $ \S 7 $ Exercícios  

✏️[__Exercício 6-07.1__](#exercicio-6-07.1) Customização

Com base nas rotinas anteriores, faça as customizações sugeridas no roteiro:

**a)** implemente o cálculo do erro envolvido no cálculo da integral pelo método do trapézio. 


**b)** adapte as rotinas para trabalhar com dados discretos, isto é, apenas com vetores `x` e `fx`, sem definir uma função algébrica. 


**c)** registre os resultados em uma tabela comparando: número de subintervalos, passo $h$, aproximação numérica e erro absoluto.

In [ ]:
# Espaço para customizações do Exercício 4.
# Sugestáo: crie uma função comparar_integracao(f, a, b, lista_n, valor_exato).

✏️[__Exercício 6-07.2__](#exercicio-6-07.2)
(a) Determine o menor $ N $ (par) para que se possa garantir que o erro envolvido na aproximação fornecida pela regra de Simpson com $ N $ subdivisões para a integral
\begin{equation*}%\label{E:}
    \int_0^{\pi} \cos x\,dx
\end{equation*}
seja menor que $ \varepsilon = 10^{-3} $ em valor absoluto.

(b) Verifique se o erro cometido para o valor de $ N $ encontrado no item (a) possui de fato módulo menor que $ \varepsilon $ calculando o valor exato da integral e sua a aproximação através da regra de Simpson.